# Installation Scripts

In [35]:
!pip install pandas numpy torch spacy nltk joblib gdown
!python -m  spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 52.5 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Imports

In [36]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
import pandas as pd
import numpy as np
import spacy
from spacy.symbols import ORTH, POS
from collections import defaultdict, Counter
import math
from nltk import ngrams
import joblib
import json
import gdown

# Constants

In [37]:
# Replace with the path of your dataset directory
INPUT_DATASET_PREFIX = "/kaggle/input/datasets/abdullahmsweesi"

# Replace with the path of your test input data file
TEST_DATA_PATH = f"{INPUT_DATASET_PREFIX}/av-data/test.csv"

# Replace with the path of your output directory
OUTPUT_PREFIX = "/kaggle/working"

# These paths will be used to store necessary data downloaded from the cloud
VOCAB_PATH = f"{OUTPUT_PREFIX}/vocab.json"
EMBEDDINGS_PATH = f"{OUTPUT_PREFIX}/reduced_embeddings.json"
MODEL_PATH = f"{OUTPUT_PREFIX}/best_model.pt"
SCALER_PATH = f"{OUTPUT_PREFIX}/style_scaler.pkl"
PREDICTIONS_PATH = f"{OUTPUT_PREFIX}/test_predictions.csv"

nlp = spacy.load("en_core_web_md", disable=["ner", "lemmatizer"])

MAX_LEN = 256        # Max sequence length (in tokens)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data Loading

In [38]:
test_df = pd.read_csv(TEST_DATA_PATH, encoding="utf-8")

In [39]:
# Download large files from the cloud

cloud_files = {
    MODEL_PATH: "13x43HGyn8Rx2BIATovEJzMeT7XUTjhgG",
    EMBEDDINGS_PATH: "17rFP_70pke2ynrLT_ryzS-KKVxzbKnbf",
    VOCAB_PATH: "1U-sNRiCwe5WkAbqs6s48CMPSpQw9u_UO",
    SCALER_PATH: "16C-RxvSCvaHrya89Tj59tUHWp8c41TSa"
}

for path, file_id in cloud_files.items():
    gdown.download(f"https://drive.google.com/uc?id={file_id}", path, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=13x43HGyn8Rx2BIATovEJzMeT7XUTjhgG
From (redirected): https://drive.google.com/uc?id=13x43HGyn8Rx2BIATovEJzMeT7XUTjhgG&confirm=t&uuid=7ef75231-8fb5-4e96-b21b-5b328600f354
To: /kaggle/working/best_model.pt
100%|██████████| 78.6M/78.6M [00:00<00:00, 88.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=17rFP_70pke2ynrLT_ryzS-KKVxzbKnbf
From (redirected): https://drive.google.com/uc?id=17rFP_70pke2ynrLT_ryzS-KKVxzbKnbf&confirm=t&uuid=0d453ad7-2435-4fc4-b13e-42ea69bd4bdd
To: /kaggle/working/reduced_embeddings.json
100%|██████████| 60.0M/60.0M [00:00<00:00, 89.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1U-sNRiCwe5WkAbqs6s48CMPSpQw9u_UO
To: /kaggle/working/vocab.json
100%|██████████| 930k/930k [00:00<00:00, 81.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=16C-RxvSCvaHrya89Tj59tUHWp8c41TSa
To: /kaggle/working/style_scaler.pkl
100%|██████████| 8.80k/8.80k [00:00<00:00, 12.2MB/s]


# Stylometric feature extraction

In [40]:
TOP_FUNCTION_WORDS = set(word.lower() for word in [               # lower-cased top 30 function words
    "the", "of", "and", "to", "a", "in", 
    "is", "you", "that", "was", "he", "his", 
    "for", "it", "with", "as", "on", "are",
    "be", "at", "by", "they", "this", "from", 
    "i", "have", "or", "had", "not", "she"
])
SORTED_TOP_FUNCTION_WORDS = sorted(list(TOP_FUNCTION_WORDS))

FUNCTION_TAGS = {                                                 # Universal POS tags corresponding to function words
    "ADP", "AUX", "CONJ", "CCONJ", 
    "DET", "PART", "PRON", "SCONJ"
}
CONTENT_TAGS = {                                                  # Universal POS tags corresponding to content words
    "ADJ", "ADV", "INTJ", "NOUN", 
    "NUM", "PROPN", "VERB"
}

NON_WORD_TAGS = {"NUM", "PUNCT", "SYM", "X"}                      # Universal POS tags corresponding to non-word tokens

ALL_NN_FORMS = {"NN", "NNS", "NNP", "NNPS"}                       # Penn Treebank POS tags corresponding to noun forms
ALL_JJ_FORMS = {"JJ", "JJR", "JJS"}                               # Penn Treebank POS tags corresponding to adjective forms
ALL_RB_FORMS = {"RB", "RBR", "RBS"}                               # Penn Treebank POS tags corresponding to adverb forms
ALL_VB_FORMS = {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"}          # Penn Treebank POS tags corresponding to verb forms
TOP_PTB_PUNCT_TAGS = {".", ",", ":"}                              # Penn Treebank POS tags corresponding to common punctuation symbols

def get_top_pos_trigrams():
    """
    Assembles the most common POS trigrams
    """
    trigrams = set()

    for nn in ALL_NN_FORMS:
        trigrams.add((nn, "IN", "DT"))                             # 'NOUN', IN, DT
        trigrams.add(("IN", "DT", nn))                             # IN, DT, 'NOUN'
        trigrams.add(("DT", nn, "IN"))                             # DT, 'NOUN', IN
        for punct in TOP_PTB_PUNCT_TAGS:
            trigrams.add(("DT", nn, punct))                        # DT, 'NOUN', 'PUNCT'
        for nn2 in ALL_NN_FORMS:
            trigrams.add(("DT", nn, nn2))                          # DT, 'NOUN', 'NOUN'

    for jj in ALL_JJ_FORMS:
        trigrams.add(("IN", "DT", jj))                             # IN, DT, 'ADJ'
        for nn in ALL_NN_FORMS:
            trigrams.add(("DT", jj, nn))                           # DT, 'ADJ', 'NOUN'
            trigrams.add((jj, nn, "IN"))                           # 'ADJ', 'NOUN', IN
            for punct in TOP_PTB_PUNCT_TAGS:
                trigrams.add((jj, nn, punct))                      # 'ADJ', 'NOUN', 'PUNCT'
            for rb in ALL_RB_FORMS:
                trigrams.add((rb, jj, nn))                         # 'ADV', 'ADJ', 'NOUN'

    for vb in ALL_VB_FORMS:
        trigrams.add(("PRP", vb, "DT"))                            # PRP, 'VERB', DT
        trigrams.add(("PRP", vb, "IN"))                            # PRP, 'VERB', IN

    return trigrams
                
TOP_POS_TRIGRAMS = get_top_pos_trigrams()
SORTED_TOP_POS_TRIGRAMS = sorted(list(TOP_POS_TRIGRAMS))

In [41]:
def assemble_stylometric_features(
    f_count, c_count,
    top_function_word_freqs,
    lexical_count,
    unique_words,
    conjunction_count,
    long_word_count,
    oov_count, non_oov_count,
    word_length_sum,
    pronoun_freqs,
    mean_sentence_length,
    std_sentence_length,
    special_punct_freqs,
    total_punct_count,
    top_pos_trigram_freqs,
    total_pos_trigram_count,
):
    """
    Extracts stylometric features from language statistics
    """
    feature_vector = []
    
    # the (normalised) frequencies of each of the top 30 function words [30 features]
    for word in SORTED_TOP_FUNCTION_WORDS:
        feature_vector.append(top_function_word_freqs[word] / (f_count + 1e-9))

    # the (normalised) ratio of function words to content words [1 feature]
    feature_vector.append(np.clip(f_count / (c_count + 1e-9), 0, 5))

    # the (normalised) type-token ratio (i.e proportion of non-unique words) [1 feature]
    feature_vector.append(len(unique_words) / (lexical_count + 1e-9))

    # the (normalised) count of conjunctions (e.g. and, but) [1 feature]
    feature_vector.append(conjunction_count / (f_count + 1e-9))

    # the (normalised) count of long words (>6 chars) [1 feature]
    feature_vector.append(long_word_count / (lexical_count + 1e-9))

    # the (normalised) ratio of OOV words to non-OOV words [1 feature]
    feature_vector.append(np.clip(oov_count / (non_oov_count + 1e-9), 0, 5))

    # the (normalised) ratio of first- and second-person pronouns to third-person pronouns [1 feature]
    feature_vector.append(np.clip(pronoun_freqs["12"] / (pronoun_freqs["3"] + 1e-9), 0, 5))

    # the (normalised) ratio of first-person singular pronouns to first-person plural pronouns [1 feature]
    feature_vector.append(np.clip(pronoun_freqs["1S"] / (pronoun_freqs["1P"] + 1e-9), 0, 5))

    # the mean word length [1 feature]
    feature_vector.append(np.clip(word_length_sum / (lexical_count + 1e-9), 0, 20))

    # the mean sentence length [1 feature]
    feature_vector.append(np.clip(mean_sentence_length, 0, 100))

    # the sentence length standard deviation [1 feature]
    feature_vector.append(np.clip(std_sentence_length, 0, 100))

    # the (normalised) frequencies of punctuation symbols in ';,.-?!' [6 features]
    for symbol in ";,.-?!":
        feature_vector.append(special_punct_freqs[symbol] / (total_punct_count + 1e-9))

    # the (normalised) ratios of punctuation symbol pairs of interest [6 features]
    punctuation_pairs = [
        (",", ";"), (",", "-"),
        (";", "-"), ("?", "."),
        (",", "."), ("!", ".")
    ]
    for s1, s2 in punctuation_pairs:
        feature_vector.append(np.clip(special_punct_freqs[s1] / (special_punct_freqs[s2] + 1e-9), 0, 5))

    # the (normalised) counts of the top POS trigrams [119 features]
    for tri in SORTED_TOP_POS_TRIGRAMS:
        feature_vector.append(top_pos_trigram_freqs[tri] / (total_pos_trigram_count + 1e-9))

    return np.array(feature_vector)

In [42]:
def extract_stylometric_features(document):
    """
    Processes a spaCy document and returns a 203-dimensional feature vector
    """
    f_count, c_count = 0, 0                          # counts of function and content words
    top_function_word_freqs = defaultdict(int)       # frequencies of top function words
    lexical_count = 0                                # count of all words
    unique_words = set()                             # set of all words (no duplicates)
    conjunction_count = 0                            # count of all conjunction words
    long_word_count = 0                              # long word (>6 chars) count
    oov_count, non_oov_count = 0, 0                  # counts of oov and vocabulary words
    word_length_sum = 0                              # sum of all word lengths
    pronoun_freqs = {                                # frequencies of various pronoun types
        "1S": 0,                             # first-person singular
        "1P": 0,                             # first-person plural
        "12": 0,                             # first- and second-person
        "3": 0                               # third-person
    }
    special_punct_freqs = defaultdict(int)           # frequencies of each of ";,.-?!"
    total_punct_count = 0                            # total punctuation count
    top_pos_trigram_freqs = defaultdict(int)         # frequencies of the top POS trigrams
    total_pos_trigram_count = 0                      # total POS trigram count

    # loop over all tokens
    for token in document:
        if token.is_space:
            continue

        text_lower = token.text.lower()
        universal_pos = token.pos_

        # word statistics
        if universal_pos not in NON_WORD_TAGS:
            unique_words.add(text_lower)
            lexical_count += 1
            word_length_sum += len(token.text)

            if text_lower in TOP_FUNCTION_WORDS:
                top_function_word_freqs[text_lower] += 1

            if universal_pos in FUNCTION_TAGS:
                f_count += 1
            elif universal_pos in CONTENT_TAGS:
                c_count += 1

            if universal_pos in ["CONJ", "CCONJ"]:
                conjunction_count += 1
            
            if len(token.text) > 6:
                long_word_count += 1

            if token.is_oov:
                oov_count += 1
            else:
                non_oov_count += 1

        # punctuation statistics
        if universal_pos == "PUNCT":
            total_punct_count += 1
            if token.text in ";,.-?!":
                special_punct_freqs[token.text] += 1

        # personal pronoun statistics
        if universal_pos == "PRON" and "Prs" in token.morph.get("PronType"):
            person = token.morph.get("Person")
            num = token.morph.get("Number")
            if "1" in person:
                pronoun_freqs["12"] += 1
                if "Sing" in num:
                    pronoun_freqs["1S"] += 1
                else:
                    pronoun_freqs["1P"] += 1
            elif "2" in person:
                pronoun_freqs["12"] += 1
            elif "3" in person:
                pronoun_freqs["3"] += 1

    # extract Penn Treebank POS tags
    ptb_tags = [token.tag_ for token in document]

    # loop over all POS trigrams
    for tri in ngrams(ptb_tags, 3):
        total_pos_trigram_count += 1
        if tri in TOP_POS_TRIGRAMS:
            top_pos_trigram_freqs[tri] += 1

    # sentence statistics
    sentence_lengths = [len([t for t in s if not t.is_space]) for s in document.sents]
    mean_sentence_length = np.mean(sentence_lengths) if sentence_lengths else 0
    if len(sentence_lengths) > 1:
        variance = sum((length - mean_sentence_length)**2 for length in sentence_lengths) / len(sentence_lengths)
        std_sentence_length = math.sqrt(variance)
    else:
        std_sentence_length = 0

    # final feature assembly
    return assemble_stylometric_features(
        f_count, c_count,
        top_function_word_freqs,
        lexical_count,
        unique_words,
        conjunction_count,
        long_word_count,
        oov_count, non_oov_count,
        word_length_sum,
        pronoun_freqs,
        mean_sentence_length,
        std_sentence_length,
        special_punct_freqs,
        total_punct_count,
        top_pos_trigram_freqs,
        total_pos_trigram_count,
    )

In [43]:
def process_input_stylometry(df):
    """
    Extracts two sets of stylometric features for each row in the dataframe
    """
    feature_vectors_1 = []
    feature_vectors_2 = []

    for document in nlp.pipe(df["text_1"], batch_size=256, n_process=-1, disable=["ner", "lemmatizer"]):
        feature_vectors_1.append(extract_stylometric_features(document))

    for document in nlp.pipe(df["text_2"], batch_size=256, n_process=-1, disable=["ner", "lemmatizer"]):
        feature_vectors_2.append(extract_stylometric_features(document))
        
    return np.array(feature_vectors_1), np.array(feature_vectors_2)

In [44]:
test_style_1, test_style_2 = process_input_stylometry(test_df)

In [45]:
# Different features can have different ranges, so we need to scale them

combined_test_style = np.vstack([test_style_1, test_style_2])

# Scaler based only on training data to prevent data leakage

scaler = joblib.load(SCALER_PATH)

# Scale all features
test_style_scaled_1 = scaler.transform(test_style_1)
test_style_scaled_2 = scaler.transform(test_style_2)

# Compute the difference
test_style_diff = np.abs(test_style_scaled_1 - test_style_scaled_2)

print(test_style_diff.shape)

(5985, 203)


# Word Embedding Generation

In [ ]:
class MyEmbedder:
    def __init__(self):
        """
        Initialise vocabulary and embeddings
        """
        with open(VOCAB_PATH, "r") as f:
            self.vocab = json.load(f)
        self.embedding_matrix = torch.load(EMBEDDINGS_PATH)

    def tokenize(self, text):
        """
        Break down a piece of text into a list of lowercased tokens
        """
        return [token.text.lower() for token in nlp.tokenizer(str(text))]

    def text_to_indices(self, text):
        """
        Represent a piece of text as a fixed-length list of vocabulary indices
        """
        tokens = self.tokenize(text)[:MAX_LEN]
        indices = [self.vocab.get(t, self.vocab.get("<UNK>", 1)) for t in tokens]
        padding = [0] * (MAX_LEN - len(indices))
        return indices + padding

# Data Preparation

In [47]:
class AVDataset(Dataset):
    def __init__(self, df, embedder, style_diff):
        assert len(df) == len(style_diff), "df size is not the same as style_diff"

        self.ids = df["id"] if "id" in df.columns else np.arange(len(df))
        self.text_1 = df["text_1"].values
        self.text_2 = df["text_2"].values
        self.embedder = embedder
        self.style_diff = style_diff

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        # no label since it's the test set
        return {
            "id": self.ids[index],
            "text_1": torch.tensor(self.embedder.text_to_indices(self.text_1[index])),
            "text_2": torch.tensor(self.embedder.text_to_indices(self.text_2[index])),
            "style_diff": torch.tensor(self.style_diff[index], dtype=torch.float32),
        }

In [48]:
my_embedder = MyEmbedder()

test_dataset = AVDataset(test_df, my_embedder, test_style_diff)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Model Configuration

In [49]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, h):
        attention_weights = self.attention(h)
        attention_weights = torch.softmax(attention_weights, dim=1)

        return torch.sum(attention_weights * h, dim=1)

In [50]:
class SiameseNN(nn.Module):
    def __init__(
        self, 
        embedding_matrix, 
        lstm_hidden_dim=256, 
        semantic_output_dim=64, 
        feature_config={"u_v": True, "diff": True, "product": False},
        lstm_layers=3, 
        style_input_dim=203, 
        style_output_dim=128, 
        final_output_dim=32,
        dropout=0.4
    ):
        super().__init__()
        self.feature_config = feature_config

        semantic_input_dim = lstm_hidden_dim * 2 * 2      # for bidirectional LSTM and 2 poolings

        multiplier = 0
        if feature_config.get("u_v"):
            multiplier += 2
        if feature_config.get("diff"):
            multiplier += 1
        if feature_config.get("product"):
            multiplier += 1
        
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)

        self.spatial_dropout = nn.Dropout2d(0.2)
        
        self.lstm = nn.LSTM(
            300,
            lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0
        )

        self.attention = AttentionPooling(lstm_hidden_dim * 2)

        # Semantic NN
        self.semantic_fc = nn.Sequential(
            nn.Linear(semantic_input_dim * multiplier, semantic_output_dim),          # input dim based on feature combination
            nn.ReLU(),
            nn.BatchNorm1d(semantic_output_dim),
            nn.Dropout(dropout)
        )

        # Stylometric NN
        self.style_fc = nn.Sequential(
            nn.Linear(style_input_dim, style_output_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(semantic_output_dim + style_output_dim, final_output_dim),
            nn.ReLU(),
            nn.Linear(final_output_dim, 1)             # produce one raw logit
        )

    def encode(self, indices):
        embedded = self.embedding(indices)
        embedded = self.spatial_dropout(
            embedded.unsqueeze(1).transpose(1, 3)       # add dimension and place embedding channel in position
        ).transpose(1, 3).squeeze(1)                    # readjust embedding channel and remove extra dimension
        
        h, _ = self.lstm(embedded)
        
        attention_pool = self.attention(h)
        max_pool, _ = torch.max(h, dim=1)
        return torch.cat([attention_pool, max_pool], dim=1)
    
    def forward(self, text_1, text_2, style_diff):
        u = self.encode(text_1)
        v = self.encode(text_2)

        # Process semantic features (different combinations can be toggled on/off)
        features = []
        if self.feature_config.get("u_v"):
            features.extend([u, v])
        if self.feature_config.get("diff"):
            features.append(torch.abs(u - v))
        if self.feature_config.get("product"):
            features.append(u * v)
            
        semantic_combined = torch.cat(features, dim=1)
        
        semantic_output = self.semantic_fc(semantic_combined)
        style_output = self.style_fc(style_diff)

        # Late fusion
        output = torch.cat((semantic_output, style_output), dim=1)
        
        return self.classifier(output)

In [51]:
# Model initialisation

model = SiameseNN(
    my_embedder.embedding_matrix, 
    lstm_hidden_dim=256,
    semantic_output_dim=64,
    feature_config={"u_v": True, "diff": True, "product": False},
    lstm_layers=3,
    style_output_dim=128,
    final_output_dim=32,
    dropout=0.4
).to(device)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

if hasattr(model, "module"):
  model.module.load_state_dict(torch.load(MODEL_PATH, map_location=device))
else:
  model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

<All keys matched successfully>

# Model Evaluation

In [52]:
test_predictions = []
model.eval()

with torch.no_grad():
    for batch in test_loader:
        test_text_1 = batch["text_1"].to(device)
        test_text_2 = batch["text_2"].to(device)
        test_style_diff = batch["style_diff"].to(device)

        with autocast("cuda" if torch.cuda.is_available() else "cpu"):
            outputs = model(test_text_1, test_text_2, test_style_diff)

        predictions = (torch.sigmoid(outputs) > 0.5).float()
        test_predictions.extend(predictions.cpu().numpy().flatten())
    

test_results_df = test_df.copy()
test_results_df["predicted_label"] = test_predictions

In [54]:
def save_predictions(predictions, prefix):
    predictions.to_csv(
        PREDICTIONS_PATH,
        index=False,
        header=False
    )

In [55]:
# save validation predictions
save_predictions(test_results_df["predicted_label"], "test_predictions.csv")